In [40]:
import argparse
import os
import sys
from typing import Callable, Dict, List, Tuple

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform, pdist

from helpers import (
    HCP_DIR, N_PARCELS, TASKS, TASK_BINARY,
    load_single_timeseries, load_condition_trial_frames, parse_stats,
    RT_KEYS, ACC_KEYS,
)


# Trait definitions

Each trait is a function taking:
- the per-subject brain matrix `B[task, cls, parcel]` (shape: 7 x 2 x 360),
- the per-subject behaviour dict `beh = {task: {'rt': float, 'acc': float}}`
- the parcel-network labels `nets[360]`.

It returns a single scalar score for one subject.
Scores are z-scored across the cohort afterwards.

In [41]:
subjects = np.loadtxt(os.path.join(HCP_DIR, "subjects_list.txt"), dtype=str).tolist()
print(f"Subjects: {len(subjects)}")

regions = np.load(f"{HCP_DIR}/regions.npy").T
names, nets = regions[0], regions[1]


Subjects: 100


In [42]:
def _net_mean(B_slice: np.ndarray, nets: np.ndarray, target_nets: List[str],
              region_names: np.ndarray = None, name_keywords: List[str] = None) -> float:
    """
    Mean BOLD across parcels matching given networks and/or name keywords.
    """
    mask = np.zeros(len(nets), dtype=bool)
    if target_nets:
        for n in target_nets:
            mask |= (nets == n)
    if name_keywords and region_names is not None:
        for kw in name_keywords:
            mask |= np.array([kw in str(r) for r in region_names])
    if mask.sum() == 0:
        return np.nan
    return float(B_slice[mask].mean())

In [43]:
def _beh(beh: Dict, task: str, field: str) -> float:
    if task in beh and field in beh[task]:
        return float(beh[task][field])
    return np.nan

In [44]:
# Brain task indices (must match TASKS order in helpers.py)
T = {name: i for i, name in enumerate(TASKS)}
# Class indices: 0 = condition0, 1 = condition1 (per TASK_BINARY)

In [45]:
def trait_fearfulness(B, beh, nets, names):
    # BOLD increase in fear vs neut, in orbito-affective + face areas.
    fear_act = _net_mean(B[T["EMOTION"], 1], nets, ["Orbito-Affec"], names, ["FFC"])
    neut_act = _net_mean(B[T["EMOTION"], 0], nets, ["Orbito-Affec"], names, ["FFC"])
    return fear_act - neut_act

In [46]:
def trait_reward_sensitivity(B, beh, nets, names):
    # BOLD increase in win vs loss in OFC / affective network.
    win_act = _net_mean(B[T["GAMBLING"], 1], nets, ["Orbito-Affec"], names, ["OFC"])
    loss_act = _net_mean(B[T["GAMBLING"], 0], nets, ["Orbito-Affec"], names, ["OFC"])
    return win_act - loss_act

In [47]:
def trait_loss_chasing(B, beh, nets, names):
    # High orbito-affective BOLD on loss trials AND minimal win>loss differentiation.
    # Proxy for gambling-vulnerability: brain reacts strongly to losses but doesn't
    # distinguish them from wins (which empirically tracks problem-gambling tendency
    # in some HCP-adjacent studies).
    loss_act = _net_mean(B[T["GAMBLING"], 0], nets, ["Orbito-Affec"], names, ["OFC"])
    rs = trait_reward_sensitivity(B, beh, nets, names)
    return loss_act - rs

In [48]:
def trait_working_memory(B, beh, nets, names):
    # Multiple-demand recruitment under load (2bk > 0bk) + behavioural performance
    # in cognitively demanding tasks. Standard "fluid intelligence" proxy.
    md_2bk = _net_mean(B[T["WM"], 1], nets,
                       ["Frontopariet", "Cingulo-Oper", "Dorsal-atten"])
    md_0bk = _net_mean(B[T["WM"], 0], nets,
                       ["Frontopariet", "Cingulo-Oper", "Dorsal-atten"])
    rel_acc = _beh(beh, "RELATIONAL", "acc")
    # Brain contrast and accuracy on z-scale themselves combine after cohort z-score.
    components = [c for c in [md_2bk - md_0bk, rel_acc] if np.isfinite(c)]
    return float(np.mean(components)) if components else np.nan

In [49]:
def trait_cognitive_control(B, beh, nets, names):
    # Speed/accuracy trade-off proxy: high accuracy + slower RT -> cautious + careful.
    rel_acc = _beh(beh, "RELATIONAL", "acc")
    rel_rt = _beh(beh, "RELATIONAL", "rt")
    if np.isfinite(rel_acc) and np.isfinite(rel_rt):
        # cautiousness ~ acc * rt (both high = cautious + correct)
        return rel_acc * rel_rt
    return np.nan

In [50]:
def trait_social_cognition(B, beh, nets, names):
    # Mentalising-vs-random contrast in TPJ/STS/temporal-pole hubs.
    ment_act = _net_mean(B[T["SOCIAL"], 1], nets, ["Default"],
                         names, ["TPOJ", "STS", "TGd"])
    rnd_act = _net_mean(B[T["SOCIAL"], 0], nets, ["Default"],
                        names, ["TPOJ", "STS", "TGd"])
    rt_diff = _beh(beh, "SOCIAL", "rt")  # we don't have per-condition RT here
    contrast = ment_act - rnd_act
    return contrast if np.isfinite(contrast) else np.nan

In [51]:
def trait_language_engagement(B, beh, nets, names):
    # Left-lateralised language-network recruitment during math vs story.
    # Math engages verbal-numerical processing; story is more passive.
    left_mask = np.array([str(r).startswith("L_") for r in names])
    lang_net = (nets == "Language") & left_mask
    if lang_net.sum() == 0:
        return np.nan
    math_act = float(B[T["LANGUAGE"], 1, lang_net].mean())
    story_act = float(B[T["LANGUAGE"], 0, lang_net].mean())
    return math_act - story_act

In [52]:
TRAITS: Dict[str, Tuple[Callable, str]] = {
    "fearfulness":          (trait_fearfulness,
                              "Fear>Neutral activation in orbito-affective + face areas"),
    "reward_sensitivity":   (trait_reward_sensitivity,
                              "Win>Loss activation in orbitofrontal cortex"),
    "loss_chasing":         (trait_loss_chasing,
                              "Strong loss activation minus reward-sensitivity"),
    "working_memory":       (trait_working_memory,
                              "Multiple-demand network load contrast + RELATIONAL accuracy"),
    "cognitive_control":    (trait_cognitive_control,
                              "RELATIONAL accuracy × RT (cautious correctness)"),
    "social_cognition":     (trait_social_cognition,
                              "Mental>Random activation in TPJ/STS/temporal pole"),
    "language_engagement":  (trait_language_engagement,
                              "Math>Story activation in left language network"),
}

# Build per-subject brain & behaviour matrices

In [53]:
def build_subject_brain(subjects: List[str], verbose: bool = True) -> Tuple[np.ndarray, List[Dict]]:
    """Returns (B[n_sub, 7, 2, 360], behav[list of dict per subject])"""
    n = len(subjects)
    B = np.zeros((n, len(TASKS), 2, N_PARCELS), dtype=np.float32)
    cnts = np.zeros((n, len(TASKS), 2), dtype=np.int64)
    behavs: List[Dict] = [dict() for _ in range(n)]

    for si, sid in enumerate(subjects):
        if verbose and si % 10 == 0:
            print(f"  subject {si+1}/{n}: {sid}")
        for ti, task in enumerate(TASKS):
            for run in (0, 1):
                try:
                    ts = load_single_timeseries(sid, task, run, remove_mean=False)
                except FileNotFoundError:
                    continue
                ts = (ts - ts.mean(axis=1, keepdims=True)) / (ts.std(axis=1, keepdims=True) + 1e-8)
                Tlen = ts.shape[1]
                for lab in (0, 1):
                    for cond in TASK_BINARY[task][str(lab)]:
                        frames_list = load_condition_trial_frames(sid, task, run, cond)
                        for fr in frames_list:
                            fr = fr[(fr >= 0) & (fr < Tlen)]
                            if fr.size == 0:
                                continue
                            B[si, ti, lab] += ts[:, fr].mean(axis=1)
                            cnts[si, ti, lab] += 1
                stats_d = parse_stats(sid, task, run)
                if stats_d:
                    rt = next((stats_d[k] * scale for k, scale in RT_KEYS[task] if k in stats_d), np.nan)
                    acc = next((stats_d[k] * scale for k, scale in ACC_KEYS[task] if k in stats_d), np.nan)
                    if task not in behavs[si]:
                        behavs[si][task] = dict(rt=[], acc=[])
                    if np.isfinite(rt):  behavs[si][task]["rt"].append(rt)
                    if np.isfinite(acc): behavs[si][task]["acc"].append(acc)

    # Normalise brain by counts
    with np.errstate(invalid="ignore"):
        for lab in (0, 1):
            for ti in range(len(TASKS)):
                c = cnts[:, ti, lab, None]
                c = np.where(c == 0, 1, c)
                B[:, ti, lab] /= c

    # Reduce behaviour to single values per (subject, task)
    for i in range(n):
        for task, d in behavs[i].items():
            behavs[i][task] = dict(
                rt=float(np.mean(d["rt"])) if d["rt"] else np.nan,
                acc=float(np.mean(d["acc"])) if d["acc"] else np.nan,
            )

    return B, behavs

# Compute traits

In [54]:
def compute_traits(B: np.ndarray, behavs: List[Dict], nets: np.ndarray,
                   names: np.ndarray) -> pd.DataFrame:
    n_sub = B.shape[0]
    rows = []
    for i in range(n_sub):
        row = {}
        for trait_name, (fn, _desc) in TRAITS.items():
            try:
                row[trait_name] = fn(B[i], behavs[i], nets, names)
            except Exception as e:
                row[trait_name] = np.nan
        rows.append(row)
    T_df = pd.DataFrame(rows)
    # Mean-impute then z-score across cohort
    T_df = T_df.fillna(T_df.mean())
    T_z = (T_df - T_df.mean()) / (T_df.std(ddof=0) + 1e-12)
    return T_z

# Visualization

In [55]:
def make_html(T_z: pd.DataFrame, subjects: List[str], highlight: str,
              scatter_x: str, scatter_y: str, out_path: str):
    trait_names = list(T_z.columns)
    n_traits = len(trait_names)
    n_sub = len(subjects)

    ### Panel layout
    # row 1 col 1-2 : radar (single subject + all-subjects overlay)
    # row 2 col 1   : heatmap
    # row 2 col 2   : scatter (2 chosen traits)
    # row 3 col 1   : correlation matrix
    fig = make_subplots(
        rows=3, cols=2,
        specs=[
            [{"type": "polar"}, {"type": "polar"}],
            [{"type": "heatmap"}, {"type": "scatter"}],
            [{"type": "heatmap", "colspan": 2}, None],
        ],
        subplot_titles=[
            f"Highlight: subject {highlight}" if highlight else "Select a subject (--highlight)",
            "All subjects overlay (faint = single subject; bold red = highlight)",
            "Heatmap: subjects × traits (clustered)",
            f"Scatter: {scatter_x} vs {scatter_y}",
            "Trait correlation matrix",
            None,
        ],
        vertical_spacing=0.08, horizontal_spacing=0.10,
        row_heights=[0.35, 0.35, 0.30],
    )

    ### 1a: highlighted-subject radar
    if highlight in subjects:
        hi_idx = subjects.index(highlight)
        vals = T_z.iloc[hi_idx].tolist()
        # close the polygon
        theta = trait_names + [trait_names[0]]
        r = vals + [vals[0]]
        fig.add_trace(go.Scatterpolar(
            r=r, theta=theta, fill="toself",
            name=str(highlight), line_color="crimson",
        ), row=1, col=1)
    else:
        # show cohort mean (all zeros) as a placeholder
        theta = trait_names + [trait_names[0]]
        r = [0] * (n_traits + 1)
        fig.add_trace(go.Scatterpolar(
            r=r, theta=theta, fill="toself", name="cohort mean",
            line_color="grey",
        ), row=1, col=1)

    ### 1b: all subjects overlay
    for i, sid in enumerate(subjects):
        vals = T_z.iloc[i].tolist()
        is_hi = (sid == highlight)
        fig.add_trace(go.Scatterpolar(
            r=vals + [vals[0]],
            theta=trait_names + [trait_names[0]],
            mode="lines",
            line=dict(color="crimson" if is_hi else "rgba(50,50,150,0.10)",
                      width=2 if is_hi else 1),
            name=str(sid),
            showlegend=False,
            hoverinfo="text",
            text=[f"{sid}<br>" + "<br>".join(f"  {t}: {v:+.2f}σ"
                                             for t, v in zip(trait_names, vals))]
                  * (n_traits + 1),
        ), row=1, col=2)

    # set polar ranges symmetrically around 0
    vmax = float(np.nanmax(np.abs(T_z.values))) * 1.05
    for polar_key in ("polar", "polar2"):
        fig.layout[polar_key].update(
            radialaxis=dict(range=[-vmax, vmax], tickvals=[-2, -1, 0, 1, 2],
                            ticktext=["−2σ", "−1σ", "0", "+1σ", "+2σ"],
                            gridcolor="lightgrey"),
            angularaxis=dict(gridcolor="lightgrey"),
            bgcolor="white",
        )

    ### 2a: hierarchically clustered heatmap
    D = pdist(T_z.values, metric="euclidean")
    Z = linkage(D, method="average")
    order = leaves_list(Z)
    fig.add_trace(go.Heatmap(
        z=T_z.values[order],
        x=trait_names,
        y=[str(subjects[i]) for i in order],
        colorscale="RdBu_r", zmid=0,
        colorbar=dict(title="z-score", x=0.46, y=0.51, len=0.30),
        hovertemplate="subject %{y}<br>%{x}: %{z:+.2f}σ<extra></extra>",
    ), row=2, col=1)

    ### 2b: 2-trait scatter
    if scatter_x in trait_names and scatter_y in trait_names:
        x = T_z[scatter_x].values
        y = T_z[scatter_y].values
        # colour by 3rd-trait if available, else by distance from origin
        colors = np.sqrt(x ** 2 + y ** 2)
        hover = [f"{sid}<br>{scatter_x}: {x[i]:+.2f}σ<br>{scatter_y}: {y[i]:+.2f}σ"
                 for i, sid in enumerate(subjects)]
        marker_colors = ["crimson" if subjects[i] == highlight else "steelblue"
                         for i in range(n_sub)]
        marker_size = [14 if subjects[i] == highlight else 8 for i in range(n_sub)]
        fig.add_trace(go.Scatter(
            x=x, y=y, mode="markers",
            marker=dict(color=marker_colors, size=marker_size,
                        line=dict(color="white", width=0.5)),
            text=hover, hoverinfo="text",
            showlegend=False,
        ), row=2, col=2)
        fig.update_xaxes(title_text=f"{scatter_x} (σ)", row=2, col=2,
                          zeroline=True, zerolinecolor="grey")
        fig.update_yaxes(title_text=f"{scatter_y} (σ)", row=2, col=2,
                          zeroline=True, zerolinecolor="grey")

    ### 3: trait correlation matrix
    corr = T_z.corr().values
    fig.add_trace(go.Heatmap(
        z=corr, x=trait_names, y=trait_names,
        colorscale="RdBu_r", zmin=-1, zmax=1,
        colorbar=dict(title="r", x=1.02, y=0.13, len=0.28),
        text=[[f"{v:+.2f}" for v in row] for row in corr],
        texttemplate="%{text}",
        textfont=dict(size=11),
        hovertemplate="%{x} ⇄ %{y}<br>r = %{z:.2f}<extra></extra>",
    ), row=3, col=1)

    fig.update_layout(
        title=dict(text="Subject trait vectors  —  each subject as a point in trait-space",
                   x=0.5, xanchor="center"),
        height=1500, width=1200,
        plot_bgcolor="white", paper_bgcolor="white",
        showlegend=False,
    )

    os.makedirs(os.path.dirname(os.path.abspath(out_path)) or ".", exist_ok=True)
    fig.write_html(out_path, include_plotlyjs="cdn", full_html=True)
    fig.show()
    print(f"  wrote {out_path}")

In [56]:
def print_extremes(T_z: pd.DataFrame, subjects: List[str], top_k: int = 3):
    print("\n=== Extreme subjects per trait (top & bottom) ===")
    for trait in T_z.columns:
        order = T_z[trait].sort_values(ascending=False)
        top = order.head(top_k)
        bot = order.tail(top_k)
        print(f"\n  {trait}")
        for label, v in top.items():
            sid = str(label) if not isinstance(label, (int, np.integer)) else subjects[int(label)]
            print(f"    most ↑  subject {sid}:  {v:+.2f}σ")
        for label, v in bot.iloc[::-1].items():
            sid = str(label) if not isinstance(label, (int, np.integer)) else subjects[int(label)]
            print(f"    most ↓  subject {sid}:  {v:+.2f}σ")

In [57]:
def plot_subject(highlight, scatter_x='fearfulness', scatter_y='reward_sensitivity', out_path=None):
    """
    Map each subject to a 7-dimensional 'trait vector' and visualise it.

    Each trait is a theory-driven linear combination of brain-activation and
    behaviour features. Higher = more of that trait. We then plot each subject
    as a vector in trait-space several ways:

    1. Per-subject radar (polar) chart   - pick one subject, see fingerprint.
    2. All-subjects radar overlay        - 100 polylines on one polar plot.
    3. Heatmap (100 subjects x 7 traits) - hierarchically clustered.
    4. Trait scatterplot                 - any two traits, hover for subject id.
    5. Trait correlation matrix          - sanity check on independence.

    All five panels are written into one interactive HTML.

    The trait definitions are deliberately simple and editable - edit TRAITS
    below to add/remove/redefine traits.
    """
    if out_path is None:
        out_path = f'figs/subject_{highlight}_traits.html'
    B, behavs = build_subject_brain(subjects)
    print("Computing traits...")
    T_z = compute_traits(B, behavs, nets, names)
    T_z.index = subjects
    
    print("\nTrait value summary (z-scores; mean=0, std=1 by construction):")
    print(T_z.describe().round(3).to_string())

    print_extremes(T_z, subjects, top_k=3)

    highlight = highlight
    if highlight not in subjects:
        print(f"  WARNING: subject {highlight} not in dataset; ignoring", file=sys.stderr)
        highlight = None

    make_html(T_z, subjects, highlight or "", scatter_x, scatter_y, out_path)
    print("\nDone. Edit TRAITS at the top of this script to change definitions.")
    

In [65]:
plot_subject("197550")

  subject 1/100: 100307
  subject 11/100: 106319
  subject 21/100: 117122
  subject 31/100: 125525
  subject 41/100: 133827
  subject 51/100: 142828
  subject 61/100: 153025
  subject 71/100: 160123
  subject 81/100: 178950
  subject 91/100: 196144
Computing traits...

Trait value summary (z-scores; mean=0, std=1 by construction):
       fearfulness  reward_sensitivity  loss_chasing  working_memory  cognitive_control  social_cognition  language_engagement
count      100.000             100.000       100.000         100.000            100.000           100.000              100.000
mean         0.000               0.000        -0.000          -0.000             -0.000            -0.000                0.000
std          1.005               1.005         1.005           1.005              1.005             1.005                1.005
min         -2.161              -2.352        -2.494          -2.352             -3.280            -2.892               -2.612
25%         -0.531              

  wrote figs/subject_197550_traits.html

Done. Edit TRAITS at the top of this script to change definitions.
